## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Artificial Neural Networks

## Inspired by Biological Neural Networks

<div style="text-align:center">
<img src="images/nn_biological.jpeg" alt="plot" width="600" />
</div>

A biological neuron receives many inputs. It weights these inputs and produces one output.

A few details:
- The output of a biological neuron is an action potential.
- The weight of each input depends on the physical properties of the synapse (the connection between neurons) and on its location.
- The total effect of the inputs on the output is similar to a **weighted sum**.
- Each neuron can receive input from thousands of other neurons (or from itself), and it can send output to thousands of other neurons.

These principles inspired Artificial Neural Networks (ANN). Biological neurons are very complex chemical and biological systems. In contrast, an ANN is a simple mathematical model that can grow very large.

## Learning goals
- see how an artificial neuron extends a linear model with a nonlinearity
- use `MLPRegressor` and `MLPClassifier` from scikit-learn
- understand epochs and the loss curve
- see why neural networks need scaled inputs
- evaluate a neural network the honest way: test data, and groups by subject

### From Linear Models to Artificial Neural Networks (ANN)
Assume that you have a linear model of the form 

$$
\hat{y} = b_0 + x_1\omega_{1} + x_2\omega_{2} + \cdots + x_m\omega_{m} 
$$


A simple way to *extend* that linear model is to apply a nonlinearity (called Activation) that transforms the output. 
$$
\hat{y} = g\left(b_0 + x_1\omega_{1} + x_2\omega_{2} + \cdots + x_m\omega_{m}\right)
$$

Where $g(\bullet)$ is a nonlinearity. Here are some common nonlinearities (x-axis are the input and y-axis are the output)
<div style="text-align:center">
<img src="images/activation_functions2.png" alt="plot" width="500" />
</div>

We have already seen this process, remember that Logistic Regression involved a linear system transformed by a sigmoid. 


In principle, an ANN is a linear system extended to include a nonlinearity. 

This basic component is called an artificial neuron. 
<div style="text-align:center">
<img src="images/ann_example.jpg" alt="plot" width="300" />
</div>

ANN are then created by stacking multiple neurons; for instance, this is an ANN with 5 neurons
<div style="text-align:center">
<img src="images/single_hidden_layer.jpg" alt="plot" width="300" />
</div>

You can also increase the number of layers; for instance, this is an ANN with two hidden layers, and different number of neurons per layer. 
<div style="text-align:center">
<img src="images/two_hidden_layer.png" alt="plot" width="300" />
</div>

ANNs can grow to be very complex, note that every line in the diagram correspond to a weight ($\omega_i$) that must be estimated using .... gradient descent.  
<div style="text-align:center">
<img src="images/large_ann.png" alt="plot" width="600" />
</div>


We will not go into the details of how the parameters are estimated. It is the same principle that we used for linear regression in module 03. There are very efficient python libraries to estimate these parameters. 

## A silly example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)   # same random noise every time you run the notebook

x_values = np.linspace(-10,10,30)
y_values = x_values*np.cos(x_values) + 1*np.random.randn(x_values.size) #a cosine with some noise
x_values = np.expand_dims(x_values,1)
y_values = np.expand_dims(y_values,1)
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, 'o-', label= 'True Relation')
plt.xlabel('x variable')
plt.ylabel('y variable')
plt.xlim([-10.1,10.1])
plt.show()

The goal is to create a simple ANN that uses the $x$ and $y$ data to predict $y$ for each value of $x$.

First, let us try a linear regression model.

In [ ]:
from sklearn.linear_model import LinearRegression

# Fit linear regression model
model = LinearRegression()
model.fit(x_values, y_values)

# Make predictions
y_pred = model.predict(x_values)

# Plot the data and predictions
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, 'o-', label='True Data')
plt.plot(x_values, y_pred, 'r-', label='Linear Regression Prediction', linewidth=2)
plt.xlabel('x variable')
plt.ylabel('y variable')
plt.xlim([-10.1, 10.1])
plt.legend()
plt.show()

A linear model does not work well here.

Let us try a neural network with one hidden layer and one neuron. We use the `tanh` activation, because it gives a smooth output.

In [ ]:
from sklearn.neural_network import MLPRegressor

# Fit neural network with one hidden layer and one neuron
nn_model = MLPRegressor(hidden_layer_sizes=(1,), max_iter=10000, random_state=42, activation = 'tanh')
nn_model.fit(x_values, y_values.ravel()) # Flatten y_values to 1D array for training

# Make predictions
y_pred_nn = nn_model.predict(x_values)

# Plot the data and predictions
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, 'o-', label='True Data')
plt.plot(x_values, y_pred_nn, 'g-', label='Neural Network Prediction (1 neuron)', linewidth=2)
plt.xlabel('x variable')
plt.ylabel('y variable')
plt.xlim([-10.1, 10.1])
plt.legend()
plt.show()

One neuron is not sufficient. Let us try one hidden layer with 10 neurons.

In [ ]:
# Fit neural network with one hidden layer and 10 neurons
nn_model_10 = MLPRegressor(hidden_layer_sizes=(10,), max_iter=10000, random_state=42, activation='tanh')
nn_model_10.fit(x_values, y_values.ravel())

# Make predictions
y_pred_nn10 = nn_model_10.predict(x_values)

# Plot the data and predictions
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, 'o-', label='True Data')
plt.plot(x_values, y_pred_nn10, 'm-', label='Neural Network Prediction (10 neurons)', linewidth=2)
plt.xlabel('x variable')
plt.ylabel('y variable')
plt.xlim([-10.1, 10.1])
plt.legend()
plt.show()

This looks much better. We basically have ten simple models (neurons) working together to predict the output. Each input ($x$) is transformed by these nonlinear models to generate an output. 

<div style="text-align:center">
<img src="images/10layers.png" alt="plot" width="600" />
</div>

In [ ]:
# A neural network with 4 hidden layers, with 4, 5, 5 and 4 neurons
nn_model_3layers = MLPRegressor(hidden_layer_sizes=(4,5,5,4), max_iter=10000, random_state=42, activation='tanh')
nn_model_3layers.fit(x_values, y_values.ravel())


# Make predictions
y_pred_nn3layers = nn_model_3layers.predict(x_values)

# Plot the data and predictions
plt.figure(figsize=(10, 6))
plt.plot(x_values, y_values, 'o-', label='True Data')
plt.plot(x_values, y_pred_nn3layers, 'm-', label='Neural Network Prediction (4 hidden layers)', linewidth=2)
plt.xlabel('x variable')
plt.ylabel('y variable')
plt.xlim([-10.1, 10.1])
plt.legend()
plt.show()

<div style="text-align:center">
<img src="images/Multilayers.png" alt="plot" width="600" />
</div>

A few details about ANNs:
1. ANNs are *universal approximators*: a sufficiently large ANN can approximate any relation between the input and the output.
2. This is true only **inside the range of the training data**. Outside this range, the prediction can be very wrong. The red areas in the next plot are outside the training range.

In [ ]:
from matplotlib.patches import Rectangle
np.random.seed(1)
x_values = np.linspace(-15,15,50)
y_values = x_values*np.cos(x_values) +  1*np.random.randn(x_values.size)

plt.gca().add_patch(Rectangle((-15, -15), 5, 30, facecolor = 'red', alpha = 0.1))
plt.gca().add_patch(Rectangle((10, -15), 5, 30, facecolor = 'red', alpha = 0.1))
plt.plot(x_values,y_values,'o-', label='Measured Data', alpha = 0.5)
out = nn_model_3layers.predict(x_values.reshape(-1,1))
plt.plot(x_values, out, label='Prediction')
plt.title('Prediction Outside Range')
plt.show()

## How the network learns: epochs and the loss curve

The network starts with random weights. Then gradient descent changes the weights in small steps to make the error (the **loss**) smaller.

- One **epoch** is one pass through all the training data.
- The **loss curve** shows the loss after each epoch. If the loss stops going down, the training is complete.

scikit-learn keeps the loss curve in the attribute `loss_curve_`. You will see the same idea in PyTorch in notebooks 05.02b and 05.02c.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(nn_model.loss_curve_, label='1 neuron')
plt.plot(nn_model_10.loss_curve_, label='10 neurons')
plt.plot(nn_model_3layers.loss_curve_, label='4 hidden layers')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Training loss (log scale)')
plt.title('Loss curves')
plt.legend()
plt.show()

print('Number of epochs used:', nn_model.n_iter_, nn_model_10.n_iter_, nn_model_3layers.n_iter_)

Very large ANNs (millions of parameters) need special methods to train well. Libraries such as **PyTorch** and TensorFlow have these methods. We will use PyTorch in the next notebooks.

## Example: ANN for classification

Neural networks are nonlinear models. They can separate classes that a line cannot separate. We make three classes in a spiral shape.

In [ ]:
np.random.seed(0)
N = 300
D = 2
K = 3
X = np.zeros((N*K,2))
Y = np.zeros(N*K, dtype = 'uint8')
for j in range(K):
    ix = range(N*j,N*(j+1))
    r = np.linspace(0.0,1,N) # radius
    t = np.linspace(j*4,(j+1)*4,N) + np.random.randn(N)*0.2 # theta
    X[ix,:] = np.c_[r*np.sin(t), r*np.cos(t)]
    Y[ix] = j
    
fig = plt.figure(figsize=(10,10))
plt.scatter(X[:, 0], X[:, 1], edgecolors='k', c=Y, s=40, cmap=plt.cm.Spectral)
plt.xlim([-1.5,1.5])
plt.ylim([-1.5,1.5])
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Classification Example')
plt.show()

As in module 04, we keep part of the data for the test. We train the models on the train set, and we measure the accuracy on the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=0, stratify=Y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

# grid of points to draw the decision regions
x_min, x_max = X[:, 0].min() - 0.2, X[:, 0].max() + 0.2
y_min, y_max = X[:, 1].min() - 0.2, X[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400))
grid = np.c_[xx.ravel(), yy.ravel()]

def plot_regions(model, title):
    Z = model.predict(grid).reshape(xx.shape)
    plt.figure(figsize=(7, 7))
    plt.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.Spectral)
    plt.scatter(X_test[:, 0], X_test[:, 1], c=Y_test, edgecolors="k", s=25, cmap=plt.cm.Spectral)
    plt.title(title + ' (points: test set)')
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

First, a logistic regression model.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(max_iter=2000, random_state=42)
logreg_model.fit(X_train, Y_train)
print(f"Logistic regression | train accuracy: {logreg_model.score(X_train, Y_train):.3f} | test accuracy: {logreg_model.score(X_test, Y_test):.3f}")
plot_regions(logreg_model, 'Logistic regression')

Logistic regression draws straight boundaries, so it does not work here. Let us try a random forest (lecture 04.02).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=300, min_samples_split=5, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, Y_train)
print(f"Random forest | train accuracy: {rf_clf.score(X_train, Y_train):.3f} | test accuracy: {rf_clf.score(X_test, Y_test):.3f}")
plot_regions(rf_clf, 'Random forest')

The random forest works. Now an ANN with two hidden layers of 5 neurons and the `tanh` activation.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_clf = MLPClassifier(hidden_layer_sizes=(5, 5), activation="tanh", solver="adam", max_iter=5000, random_state=42)
mlp_clf.fit(X_train, Y_train)
print(f"MLP | train accuracy: {mlp_clf.score(X_train, Y_train):.3f} | test accuracy: {mlp_clf.score(X_test, Y_test):.3f}")
plot_regions(mlp_clf, 'MLP (5, 5)')

The ANN also works. Compare the shape of the regions: the random forest makes regions with steps (each tree split is a vertical or horizontal line), and the ANN makes smooth regions.

## Neural networks need scaled inputs

In the spiral example, the two features have the same range (−1 to 1). Real data is different. We use the gait data from Assignments 6 and 7 (`gait_measures_PD_controls_ALL.csv`) to classify control vs. Parkinson's disease (PD). Cadence is about 50 to 170 steps/min, but the step times are about 0.1 to 2 s.

An ANN multiplies each input by a weight and adds the results. If one input is 100 times larger than the others, it controls the result, and the training becomes difficult. So we always put a `StandardScaler` before the ANN, in a `Pipeline` (as in lecture 04.03).

Each subject has more than one recording, so we use `GroupKFold` with the subject `ID` as the group (lecture 04.04).

In [ ]:
import pandas as pd
import warnings
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

gait = pd.read_csv('data/gait_measures_PD_controls_ALL.csv').dropna()
features = [c for c in gait.columns if c not in ['ID', 'class', 'avg_speed']]
X_gait, y_gait, groups = gait[features], gait['class'], gait['ID']
print(f"{len(gait)} recordings from {groups.nunique()} subjects, {len(features)} features")
print(X_gait[['cadence', 'avg_step_time', 'cv_step_time']].describe().loc[['min', 'max']])

models = {
    'MLP, no scaling': MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000, random_state=0),
    'MLP, scaled':     Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000, random_state=0))]),
    'Logistic regression, scaled': Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=5000))]),
    'Random forest':   RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
}

cv = GroupKFold(n_splits=5)
for name, model in models.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')   # the unscaled MLP gives convergence warnings
        scores = cross_val_score(model, X_gait, y_gait, groups=groups, cv=cv, scoring='balanced_accuracy')
    print(f"{name:30s} balanced accuracy = {scores.mean():.3f} ± {scores.std():.3f}")

Without scaling, the ANN is almost at chance level (balanced accuracy 0.5). With scaling, it is similar to logistic regression and random forest.

On a small tabular dataset (164 subjects, 16 features), an ANN is usually **not better** than simpler models. ANNs are most useful when there is a lot of data, or when the input is a signal or an image. We will see this with convolutional neural networks in 05.02.

## Activity

Use the gait data and the same `GroupKFold`.
1. Train the scaled MLP with `hidden_layer_sizes` equal to `(2,)`, `(20,)`, `(100,)` and `(100, 100)`. Print the mean balanced accuracy for each.
2. Does a larger network give a better result? Why, or why not?
3. Add `alpha=1.0` to the `(100, 100)` network (`alpha` is the L2 penalty, as in ridge regression). What changes?

In [ ]:
# Write your code here
